#### Optuna

- 최적의 parameter를 찾기 위한 라이브러리
- `create_study()` 라는 내장 함수를 이용하여 특정 모델의 최적의 parameter를 search
- GridSearchCV에 비하여 속도 면에서 우세
    - GridSearchCV는 parameter의 모든 조합을 fit하고 검증의 결과를 확인
    - Optuna는 확률 기반 → 모든 조합을 활용하지는 않는다.
- 조합의 수 제어
    - GridSearchCV: params로 제어
    - Optuna: n_trials 매개변수로 제어

In [2]:
# !pip install optuna

In [3]:
import optuna
from sklearn.datasets import load_iris
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, make_scorer

c:\Users\hkssn\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
X, y = load_iris(return_X_y = True)

In [5]:
# objective 함수 생성: parameter의 조합, pipe, fold, 검증 방법을 하나의 함수에 집어넣어주는 것

def objective(trial):
    # SVC 모델의 파라미터 조합을 생성
    # suggest_XXX
        # suggest_int, suggest_float: 정수, 실수 형태의 파라미터 조합 (시작값, 종료값(종료값 포함), log 매개변수)
            # log 매개변수: False 기본값, True로 변경하면 로그 스케일로 조합을 생성 (실수 형태에서 사용)
        # suggest_categorical: 특정 범주 조합
    C = trial.suggest_float('C', 1e-3, 10.0, log = True)
    gamma = trial.suggest_float('gamma', 1e-4, 1.0, log = True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    model = SVC(C = C, gamma = gamma, kernel = kernel)

    pipe = Pipeline(
        [
            ('std', StandardScaler()),
            ('clf', model)
        ]
    )

    cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
    scores = cross_val_score(pipe, X, y, cv = cv, scoring = make_scorer(f1_score, average = 'macro'))

    return scores.mean()

In [6]:
study = optuna.create_study(
    direction = 'maximize',
    study_name = 'class_ml_tuning'
)

study.optimize(
    objective, n_trials = 30, show_progress_bar = True
)

[I 2026-06-11 10:31:44,570] A new study created in memory with name: class_ml_tuning
Best trial: 4. Best value: 0.966583:  17%|█▋        | 5/30 [00:00<00:00, 29.53it/s]

[I 2026-06-11 10:31:44,622] Trial 0 finished with value: 0.9254223367455297 and parameters: {'C': 0.042024544158350406, 'gamma': 0.01068672892936225, 'kernel': 'linear'}. Best is trial 0 with value: 0.9254223367455297.
[I 2026-06-11 10:31:44,648] Trial 1 finished with value: 0.8651321398124466 and parameters: {'C': 0.003698596948490338, 'gamma': 0.09804312969438786, 'kernel': 'linear'}. Best is trial 0 with value: 0.9254223367455297.
[I 2026-06-11 10:31:44,678] Trial 2 finished with value: 0.9254223367455297 and parameters: {'C': 0.025927066567083844, 'gamma': 0.017585534806158553, 'kernel': 'linear'}. Best is trial 0 with value: 0.9254223367455297.
[I 2026-06-11 10:31:44,716] Trial 3 finished with value: 0.891698555852547 and parameters: {'C': 0.15613806506714786, 'gamma': 0.08757914084104243, 'kernel': 'rbf'}. Best is trial 0 with value: 0.9254223367455297.
[I 2026-06-11 10:31:44,742] Trial 4 finished with value: 0.9665831244778612 and parameters: {'C': 0.05516884435760017, 'gamma': 

Best trial: 4. Best value: 0.966583:  37%|███▋      | 11/30 [00:00<00:00, 29.01it/s]

[I 2026-06-11 10:31:44,807] Trial 6 finished with value: 0.8651321398124466 and parameters: {'C': 0.0052154815353534145, 'gamma': 0.03163620845614564, 'kernel': 'linear'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:44,841] Trial 7 finished with value: 0.9665831244778612 and parameters: {'C': 0.051923343968342445, 'gamma': 0.0038418768190424768, 'kernel': 'linear'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:44,877] Trial 8 finished with value: 0.8850151807481194 and parameters: {'C': 0.012638456280812753, 'gamma': 0.2280165991534357, 'kernel': 'rbf'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:44,909] Trial 9 finished with value: 0.9599331662489557 and parameters: {'C': 0.0581144164252944, 'gamma': 0.00020573148680880235, 'kernel': 'linear'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:44,954] Trial 10 finished with value: 0.8651321398124466 and parameters: {'C': 9.268030798809352, 'gamma

Best trial: 4. Best value: 0.966583:  57%|█████▋    | 17/30 [00:00<00:00, 29.80it/s]

[I 2026-06-11 10:31:45,017] Trial 12 finished with value: 0.8651321398124466 and parameters: {'C': 0.0012736890849324384, 'gamma': 0.0026542114947140276, 'kernel': 'linear'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:45,052] Trial 13 finished with value: 0.9599331662489557 and parameters: {'C': 0.41532203443771076, 'gamma': 0.002984006018494226, 'kernel': 'linear'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:45,083] Trial 14 finished with value: 0.9598997493734336 and parameters: {'C': 0.0987617455493518, 'gamma': 0.9481673549590561, 'kernel': 'linear'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:45,119] Trial 15 finished with value: 0.9661728917348785 and parameters: {'C': 1.591154469001319, 'gamma': 0.006538018041458927, 'kernel': 'linear'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:45,150] Trial 16 finished with value: 0.8848460711053253 and parameters: {'C': 0.014009216920222632, '

Best trial: 23. Best value: 0.973266:  77%|███████▋  | 23/30 [00:00<00:00, 31.16it/s]

[I 2026-06-11 10:31:45,214] Trial 18 finished with value: 0.8651321398124466 and parameters: {'C': 0.1288539742825258, 'gamma': 0.000625974988515016, 'kernel': 'rbf'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:45,247] Trial 19 finished with value: 0.8651321398124466 and parameters: {'C': 0.006660874990135579, 'gamma': 0.00010496971474791688, 'kernel': 'linear'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:45,277] Trial 20 finished with value: 0.8651321398124466 and parameters: {'C': 0.0016177546056321478, 'gamma': 0.006234204761865077, 'kernel': 'linear'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:45,302] Trial 21 finished with value: 0.9661728917348785 and parameters: {'C': 2.4198941500051387, 'gamma': 0.006149564268460307, 'kernel': 'linear'}. Best is trial 4 with value: 0.9665831244778612.
[I 2026-06-11 10:31:45,333] Trial 22 finished with value: 0.9661728917348785 and parameters: {'C': 6.199000319350298, 'g

Best trial: 23. Best value: 0.973266: 100%|██████████| 30/30 [00:00<00:00, 31.45it/s]

[I 2026-06-11 10:31:45,396] Trial 24 finished with value: 0.9732664995822891 and parameters: {'C': 0.7688421305456474, 'gamma': 0.02291783942943099, 'kernel': 'linear'}. Best is trial 23 with value: 0.9732664995822891.
[I 2026-06-11 10:31:45,425] Trial 25 finished with value: 0.9732664995822891 and parameters: {'C': 0.7753045207293681, 'gamma': 0.034375549315074035, 'kernel': 'linear'}. Best is trial 23 with value: 0.9732664995822891.
[I 2026-06-11 10:31:45,455] Trial 26 finished with value: 0.9599331662489557 and parameters: {'C': 0.684870588200468, 'gamma': 0.04486305121026648, 'kernel': 'rbf'}. Best is trial 23 with value: 0.9732664995822891.
[I 2026-06-11 10:31:45,481] Trial 27 finished with value: 0.9661728917348785 and parameters: {'C': 2.7899035351031443, 'gamma': 0.10413772630830913, 'kernel': 'linear'}. Best is trial 23 with value: 0.9732664995822891.
[I 2026-06-11 10:31:45,506] Trial 28 finished with value: 0.9663805979595453 and parameters: {'C': 0.9715111816098477, 'gamma':

In [ ]:
# 최적의 파라미터
study.best_params

{'C': 0.8829885081799306, 'gamma': 0.012693563325866724, 'kernel': 'linear'}

In [8]:
# 최적의 스코어
study.best_value

0.9732664995822891